# 02 — Detekcija sistematskih (logičkih) grešaka

Pošto smo prošli inicijalni pregled, ovde tražimo logičke nekonzistentnosti između kolona — kombinacije vrednosti koje nisu statistički čudne, ali su logički nemoguće ili bar sumnjive.

Ovakve stvari `isnull()`, `describe()` ili box-plot neće uhvatiti, jer je svaka vrednost sama za sebe sasvim u redu. Problem iskoči tek kad pogledaš odnose između kolona.

Napomena: ovo radimo pre podele na trening i test skup, pošto su u pitanju strukturne provere koje ne zavise od statistike podataka. Nemoguća kombinacija je nemoguća bez obzira na raspodelu.

## 1. Učitavanje podataka

Učitavamo isti skup kao u prethodnoj svesci i radimo istu konverziju `TotalCharges` u broj, da bismo krenuli od istog stanja podataka na kom smo završili u svesci `01`.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(f"Podaci ucitani: {df.shape[0]} redova x {df.shape[1]} kolona")

Podaci ucitani: 7043 redova x 21 kolona


## 2. Konzistentnost: PhoneService <-> MultipleLines

Logika je sledeća: ako korisnik nema telefonsku uslugu (`PhoneService = No`), onda `MultipleLines` mora biti `"No phone service"`. Bilo šta drugo (`"Yes"` ili `"No"`) nema smisla — kako bi neko imao ili nemao više telefonskih linija ako uopšte nema telefon?

Proveravamo da li ovo važi za sve redove.

In [2]:
# Izdvajamo redove gde korisnik nema telefonsku uslugu.
# df["PhoneService"] == "No" vraća seriju True/False za svaki red.
bez_telefona = df[df["PhoneService"] == "No"]

print(f"Broj korisnika bez telefonske usluge: {len(bez_telefona)}")

# Za te korisnike, prikazujemo koje vrednosti imaju u MultipleLines.
# Očekivanje: sve treba da bude "No phone service".
print("\nVrednosti MultipleLines za korisnike bez telefonske usluge:")
print(bez_telefona["MultipleLines"].value_counts())

Broj korisnika bez telefonske usluge: 682

Vrednosti MultipleLines za korisnike bez telefonske usluge:
MultipleLines
No phone service    682
Name: count, dtype: int64


### Rezultat provere

Svih 682 korisnika bez telefonske usluge ima vrednost `"No phone service"` u koloni `MultipleLines`. Znači konzistentno je, nema logičkih grešaka između ove dve kolone.

## 3. Konzistentnost: InternetService <-> dodatne internet usluge

Ista priča kao malopre: ako korisnik nema internet (`InternetService = No`), onda sve dodatne internet usluge moraju imati vrednost `"No internet service"`. Sve drugo bi bilo nemoguće — kako bi neko imao online zaštitu, backup ili streaming bez interneta?

Dodatne internet usluge su `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV` i `StreamingMovies`.

Ovaj put, dakle, proveravamo konzistentnost preko 6 kolona odjednom.

In [3]:
# Izdvajamo redove gde korisnik nema internet uslugu.
bez_interneta = df[df["InternetService"] == "No"]

print(f"Broj korisnika bez internet usluge: {len(bez_interneta)}")

# Lista dodatnih internet usluga koje proveravamo.
dodatne_usluge = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

# Za svaku od tih kolona prikazujemo raspodelu vrednosti kod korisnika bez interneta.
# Očekivanje: sve treba da bude "No internet service".
print("\nRaspodela vrednosti za korisnike bez interneta:")
for usluga in dodatne_usluge:
    print(f"\n{usluga}:")
    print(bez_interneta[usluga].value_counts())

Broj korisnika bez internet usluge: 1526

Raspodela vrednosti za korisnike bez interneta:

OnlineSecurity:
OnlineSecurity
No internet service    1526
Name: count, dtype: int64

OnlineBackup:
OnlineBackup
No internet service    1526
Name: count, dtype: int64

DeviceProtection:
DeviceProtection
No internet service    1526
Name: count, dtype: int64

TechSupport:
TechSupport
No internet service    1526
Name: count, dtype: int64

StreamingTV:
StreamingTV
No internet service    1526
Name: count, dtype: int64

StreamingMovies:
StreamingMovies
No internet service    1526
Name: count, dtype: int64


### Rezultat provere

Svih 1526 korisnika bez interneta ima vrednost `"No internet service"` u svih 6 dodatnih usluga (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`). Konzistentno je, nema logičkih grešaka oko internet usluga.

## 4. Efikasnija provera pomoću vektorizovanih operacija

Provera iz prethodne sekcije radi, ali izbaci 6 tabela. Za brz pregled je zgodnije da samo prebrojimo koliko redova krši logiku, umesto da gledamo sve.

Koristimo pandas metod `.isin()`, koji za svaki red vraća `True` ako je vrednost u zadatoj listi.

In [ ]:
# Za svaku dodatnu uslugu proveravamo koliko korisnika BEZ interneta ima
# vrednost različitu od "No internet service" (što bi bila logička greška).

print("Broj problematičnih redova po koloni:")


for usluga in dodatne_usluge:
    broj_problema = (bez_interneta[usluga] != "No internet service").sum()
    print(f"  {usluga}: {broj_problema}")

Broj problematičnih redova po koloni:
  OnlineSecurity: 0
  OnlineBackup: 0
  DeviceProtection: 0
  TechSupport: 0
  StreamingTV: 0
  StreamingMovies: 0


## 5. Provera korisnika sa `tenure = 0`

`tenure` je broj meseci koliko je korisnik u kompaniji, pa vrednost `0` znači korisnike koji su tek počeli.

Ovo proveravamo da vidimo koliko takvih „novih" korisnika ima i da eksplicitno potvrdimo ono što već znamo iz sveske `01` — da su to isti oni kod kojih fali `TotalCharges`.

In [ ]:
# Broj korisnika sa tenure = 0.
tenure_nula = df[df["tenure"] == 0]
print(f"Broj korisnika sa tenure = 0: {len(tenure_nula)}")

# Provera: da li se to poklapa sa brojem korisnika kojima fali TotalCharges?
totalcharges_nan = df[df["TotalCharges"].isnull()]
print(f"Broj korisnika sa TotalCharges = NaN: {len(totalcharges_nan)}")

# print(set(tenure_nula["customerID"]))
# print(set(totalcharges_nan["customerID"]))
isti_korisnici = set(tenure_nula["customerID"]) == set(totalcharges_nan["customerID"])
print(f"Da li su to isti korisnici? {isti_korisnici}")

Broj korisnika sa tenure = 0: 11
Broj korisnika sa TotalCharges = NaN: 11
Da li su to isti korisnici? True


### Rezultat provere

U skupu ima 11 korisnika sa `tenure = 0`. Poređenje `customerID` vrednosti potvrđuje da su to iste osobe kojima je nedostajao `TotalCharges`.

Ti korisnici su tehnički ispravni (tek su se pretplatili), ali će zbog nedostajućeg `TotalCharges` dobiti poseban tretman u narednim sveskama, nakon podele na trening/test.

---

## 6. Zaključak

Provera sistematskih grešaka pokazuje da je skup logički konzistentan.

Prošle su sve tri provere:
1. Svi korisnici bez telefonske usluge imaju `MultipleLines = "No phone service"`.
2. Svi korisnici bez interneta imaju `"No internet service"` u svih 6 dodatnih usluga.
3. Korisnici sa `tenure = 0` su ista grupa kod koje fali `TotalCharges`.

Za dalji rad ovo znači da kolone `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV` i `StreamingMovies` nose konzistentnu informaciju o tome da usluga ne postoji (`"No phone service"` / `"No internet service"`). Odatle se otvara prostor za feature engineering — možemo razmišljati o spajanju `"No"` i `"No internet service"` u istu kategoriju ili o pravljenju novih izvedenih atributa.

